# Chapter 3 Practical 04: Item-Item Collaborative Filtering

Learning objectives:
- Compute item-item similarities from user ratings.
- Predict a missing rating from similar items the user already rated.
- Compare item-item CF with user-user CF.
- Discuss why item-item CF is often easier to cache and serve.

Slide connection: item-item CF concept, item-item prediction example, and scalability.


In [1]:
# Teaching note: Load ratings and movies with a Colab-safe fallback. Local files are used first; GitHub raw CSVs are used when opened directly from GitHub.
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIRS = [
    Path("data"),
    Path("../data"),
    Path("chapter_03_collaborative_filtering/data"),
]
GITHUB_DATA_URL = "https://raw.githubusercontent.com/MehrdadJalali-AI/RecommenderSystems/main/chapter_03_collaborative_filtering/data"

def read_chapter3_csv(filename):
    for data_dir in DATA_DIRS:
        csv_path = data_dir / filename
        if csv_path.exists():
            print(f"Loaded {filename} from {csv_path}")
            return pd.read_csv(csv_path)
    url = f"{GITHUB_DATA_URL}/{filename}"
    print(f"Local file not found. Loading {filename} from GitHub raw URL.")
    return pd.read_csv(url)

ratings = read_chapter3_csv("ratings_chapter3.csv")
movies = read_chapter3_csv("movies_chapter3.csv")
ratings_named = ratings.merge(movies, on="movie_id", how="left")
# Pivot interactions into rows = users and columns = items.
rating_matrix = ratings_named.pivot_table(index="user_id", columns="title", values="rating")
rating_matrix


Loaded ratings_chapter3.csv from data/ratings_chapter3.csv
Loaded movies_chapter3.csv from data/movies_chapter3.csv


title,Blade Runner,Finding Nemo,Independence Day,Jurassic Park,Star Wars,Terminator 2,The Matrix,The Notebook,Titanic,Toy Story
user_id,,,,,,,,,,
Alice,5.0,NaN,4.0,NaN,4.0,NaN,5.0,NaN,NaN,NaN
Bob,NaN,NaN,6.0,4.0,7.0,4.0,7.0,NaN,NaN,NaN
Chris,NaN,NaN,2.0,7.0,3.0,7.0,NaN,NaN,NaN,5.0
Karen,NaN,NaN,NaN,4.0,7.0,3.0,6.0,NaN,NaN,NaN
Lynn,NaN,NaN,2.0,4.0,4.0,6.0,NaN,NaN,NaN,6.0
Nina,NaN,5.0,NaN,4.0,NaN,NaN,NaN,NaN,2.0,5.0
Omar,NaN,2.0,NaN,3.0,NaN,NaN,NaN,5.0,5.0,NaN
Sally,NaN,NaN,7.0,6.0,7.0,3.0,6.0,NaN,NaN,NaN


In [2]:
# Teaching note: Compute item-item Pearson similarities from users who rated both movies.
# Item similarity is computed from users who rated both items.
def item_pearson(matrix, item_a, item_b):
    pair = matrix[[item_a, item_b]].dropna()
    if len(pair) < 2:
        return np.nan
    if pair[item_a].std() == 0 or pair[item_b].std() == 0:
        return np.nan
    return float(np.corrcoef(pair[item_a], pair[item_b])[0, 1])

items = rating_matrix.columns
item_sim = pd.DataFrame(index=items, columns=items, dtype=float)
for a in items:
    for b in items:
        item_sim.loc[a, b] = 1.0 if a == b else item_pearson(rating_matrix, a, b)

item_sim.round(2)


title,Blade Runner,Finding Nemo,Independence Day,Jurassic Park,Star Wars,Terminator 2,The Matrix,The Notebook,Titanic,Toy Story
title,,,,,,,,,,
Blade Runner,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Finding Nemo,NaN,1.0,NaN,1.00,NaN,NaN,NaN,NaN,-1.0,NaN
Independence Day,NaN,NaN,1.00,-0.11,0.94,-0.97,0.65,NaN,NaN,NaN
Jurassic Park,NaN,1.0,-0.11,1.00,-0.45,0.39,-0.50,NaN,-1.0,-0.5
Star Wars,NaN,NaN,0.94,-0.45,1.00,-0.97,0.82,NaN,NaN,1.0
Terminator 2,NaN,NaN,-0.97,0.39,-0.97,1.00,1.00,NaN,NaN,-1.0
The Matrix,NaN,NaN,0.65,-0.50,0.82,1.00,1.00,NaN,NaN,NaN
The Notebook,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN
Titanic,NaN,-1.0,NaN,-1.00,NaN,NaN,NaN,NaN,1.0,NaN


In [3]:
# Teaching note: Inspect which movies are most similar to the target item.
target_item = "Independence Day"
item_sim[target_item].drop(target_item).sort_values(ascending=False).round(3)


title
Star Wars        0.938
The Matrix       0.655
Jurassic Park   -0.106
Terminator 2    -0.972
Blade Runner       NaN
Finding Nemo       NaN
The Notebook       NaN
Titanic            NaN
Toy Story          NaN
Name: Independence Day, dtype: float64

In [4]:
# Teaching note: Predict a user rating from similar items the user has already rated.
# Item-item prediction combines the user ratings for similar items.
def predict_item_item(matrix, user, target_item, k=3, positive_only=False):
    rated = matrix.loc[user].dropna()
    candidates = []
    for item, rating in rated.items():
        sim = item_sim.loc[target_item, item]
        if pd.isna(sim):
            continue
        if positive_only and sim <= 0:
            continue
        candidates.append({"rated_item": item, "rating": rating, "similarity": sim})
    evidence = pd.DataFrame(candidates, columns=["rated_item", "rating", "similarity"])
    evidence = evidence.sort_values("similarity", ascending=False).head(k)
    if evidence.empty:
        return np.nan, evidence
    numerator = (evidence["rating"] * evidence["similarity"]).sum()
    denominator = evidence["similarity"].abs().sum()
    return numerator / denominator if denominator else np.nan, evidence

pred, evidence = predict_item_item(rating_matrix, "Karen", "Independence Day", k=3)
print(f"Predicted Karen rating for Independence Day: {pred:.2f}")
evidence.round(3)


Predicted Karen rating for Independence Day: 5.93


,rated_item,rating,similarity
1,Star Wars,7.0,0.938
3,The Matrix,6.0,0.655
0,Jurassic Park,4.0,-0.106


In [5]:
# Teaching note: Recommend unseen movies using item-item predicted ratings.
# Recommend unseen items with the highest item-item predicted ratings.
def recommend_item_item(matrix, user, n=5, k=3):
    unseen_items = matrix.columns[matrix.loc[user].isna()]
    rows = []
    for item in unseen_items:
        pred, evidence = predict_item_item(matrix, user, item, k=k, positive_only=True)
        if not pd.isna(pred):
            rows.append({
                "user": user,
                "recommended_movie": item,
                "predicted_rating": pred,
                "similar_rated_items": ", ".join(evidence["rated_item"].tolist()),
            })
    return pd.DataFrame(rows).sort_values("predicted_rating", ascending=False).head(n)

recommend_item_item(rating_matrix, "Karen").round(2)


,user,recommended_movie,predicted_rating,similar_rated_items
2,Karen,Toy Story,7.00,Star Wars
1,Karen,Independence Day,6.59,"Star Wars, The Matrix"
0,Karen,Finding Nemo,4.00,Jurassic Park


## Challenge Lab

1. Turn `positive_only` off and inspect whether negative item similarity helps or hurts the prediction.
2. For one target item, count how many users co-rated each similar item. Which similarities are based on weak evidence?
3. Compare item-item recommendations for `Karen` and `Alice`. Which rated items explain the top recommendation?
4. Reflection: why is item-item CF often easier to cache in a production system than user-user CF?
